In [0]:
spark.version

In [0]:
%run ../tests/test_silver_layer

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import logging

In [0]:
# Setup the logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger("Enrich Taxi Data Silver Layer")

**Databricks Widgets Set Up**

In [0]:
dbutils.widgets.text("silver_taxi", "")
dbutils.widgets.text("silver_weather", "")
dbutils.widgets.text("silver_holidays", "")
dbutils.widgets.text("silver_taxi_enriched", "")


silver_taxi_data = dbutils.widgets.get("silver_taxi") or "chicago_taxi_data.silver.silver_taxi"
silver_weather_data = dbutils.widgets.get("silver_weather") or "chicago_taxi_data.silver.silver_weather"
silver_holidays_data = dbutils.widgets.get("silver_holidays") or "chicago_taxi_data.silver.silver_holidays"
silver_taxi_data_enriched = dbutils.widgets.get("silver_taxi_enriched") or "chicago_taxi_data.silver.silver_taxi_enriched"

In [0]:
df_silver_weather = spark.read.format("delta").table(silver_weather_data)
df_silver_holidays = spark.read.format("delta").table(silver_holidays_data)
df_silver_taxi = spark.read.format("delta").table(silver_taxi_data)


In [0]:
df_silver_weather.printSchema()

In [0]:
display(df_silver_weather.limit(5))

In [0]:
df_silver_holidays.printSchema()

In [0]:
display(df_silver_holidays.limit(5))

In [0]:
df_silver_taxi.printSchema()

**Join Taxi Delta Table With Weather And Holidays. Prepare Chicago Taxi Data Enriched Silver Layer**

In [0]:
# Prepare weather, taxi and holidays
# Force casting to be 100% sure
df_weather_ready = df_silver_weather.withColumn("date", F.to_date("date")) \
                                   .withColumn("hour", F.col("hour").cast("int"))\
                                   .drop("year")

df_silver_taxi = df_silver_taxi.withColumn("date", F.to_date("date")) \
                               .withColumn("hour", F.col("hour").cast("int"))

                               
df_holidays_ready = df_silver_holidays.select("date", "holiday")

# Join delta tables to enrich taxi data and fill null values in holiday column with None
df_silver_taxi_enriched = df_silver_taxi.join(F.broadcast(df_weather_ready), ["date", "hour"], "left") \
                    .join(F.broadcast(df_holidays_ready), "date", "left")\
                    .fillna({"holiday": "None"})


In [0]:
df_weather_ready = df_silver_weather.withColumn("date", F.to_date("date")) \
                                   .withColumn("hour", F.col("hour").cast("int"))\
                                   .drop("year")

In [0]:
weather_daily = (
    df_weather_ready
    .groupBy("date")
    .agg(
        F.min("temperature").alias("min_temperature"),
        F.max("temperature").alias("max_temperature"),
        F.avg("temperature").cast("int").alias("avg_temperature"),
        F.sum("precipitation").alias("precipitation"),
        F.avg("humidity").cast("int").alias("avg_humidity"),
        F.max("wind_speed").alias("max_wind_speed")
    )
)

In [0]:
display(weather_daily)

In [0]:
taxi_daily = (
    df_silver_taxi
    .groupBy(
        "date",
        "year",
        "month",
        "day",
        "is_weekend",
        "week_day"
    )
    .agg(
        # --- Demand & Revenue ---
        F.count("*").alias("total_trips"),
        F.round(F.sum("fare"), 2).alias("total_fare_revenue"),
        F.round(F.sum("tips"), 2).alias("total_tips"),
        F.round(F.sum("extras"), 2).alias("total_extra_fees"),
        F.round(F.sum("tolls"), 2).alias("total_tolls"),

        # --- Payment Type ---
        F.sum(
            F.when(
                F.col("payment_type").isin("Credit Card", "Prcard", "Mobile"),
                1
            ).otherwise(0)
        ).alias("count_digital_trips"),

        F.sum(
            F.when(F.col("payment_type") == "Cash", 1).otherwise(0)
        ).alias("count_cash_trips"),

        F.sum(
            F.when(F.col("payment_type") == "Prepaid", 1).otherwise(0)
        ).alias("count_prepaid_trips"),

        F.sum(
            F.when(
                F.col("payment_type").isin("Dispute", "No Charge", "Unknown"),
                1
            ).otherwise(0)
        ).alias("count_irregular_trips"),

        # --- Average Trip Characteristics ---
        F.round(F.avg("trip_miles"), 2).alias("avg_trip_miles")
    )
)

# # Add holidays
taxi_daily = (
    taxi_daily
    .join(F.broadcast(df_holidays_ready), "date", "left")
    .fillna({"holiday": "None"})
)

# # Join weather aggregated at daily grain
# df_gold_taxi_daily = (
#     taxi_daily
#     .join(weather_daily, "date", "left")
#     .withColumn(
#         "total_gross_revenue",
#         F.round(
#             F.col("total_fare_revenue")
#             + F.col("total_tips")
#             + F.col("total_extra_fees")
#             + F.col("total_tolls"),
#             2
#         )
#     )
#     .withColumn(
#         "tip_rate",
#         F.round(
#             F.col("total_tips") / F.col("total_fare_revenue"),
#             4
#         )
#     )
#     .withColumn(
#         "temperature_swing",
#         F.round(
#             F.col("max_temperature")
#             - F.col("min_temperature"),
#             2
#         )
#     )
#     .orderBy("date")
# )

In [0]:
display(taxi_daily.orderBy(F.col("date")))

In [0]:
df_silver_taxi_enriched.columns

In [0]:
display(df_silver_taxi_enriched.limit(5))

**Data Quality Check**

In [0]:

required_columns = ["date", "trip_id", "taxi_id", "trip_start_timestamp", "trip_end_timestamp", "trip_seconds", "trip_miles", "fare", "tips", "tolls", "extras", "trip_total", "payment_type", "company", "pickup_community_area", "dropoff_community_area", "pickup_census_tract", "dropoff_census_tract", "pickup_centroid_latitude", "pickup_centroid_longitude", "dropoff_centroid_latitude", "dropoff_centroid_longitude", "pickup_centroid_location", "dropoff_centroid_location", "holiday","hour", "temperature", "humidity", "precipitation", "wind_speed"]

try:
    logger.info("Starting Chicago Taxi Enriched Features DQ checks Silver Layer")

    validate_no_nulls(df_silver_taxi_enriched, "date")
    validate_no_nulls(df_silver_taxi_enriched, "hour")
    validate_no_nulls(df_silver_taxi_enriched, "trip_id")
    validate_no_nulls(df_silver_taxi_enriched, "temperature")
    validate_no_nulls(df_silver_taxi_enriched, "precipitation")
    validate_no_nulls(df_silver_taxi_enriched, "wind_speed")
    validate_no_nulls(df_silver_taxi_enriched, "humidity")
    validate_no_nulls(df_silver_taxi_enriched, "fare")
    validate_schema(df_silver_taxi_enriched, required_columns)
    validate_duplicates(df_silver_taxi_enriched, ["date", "hour", "trip_id"])
    #validate_temperature_range(df_silver_taxi_enriched, "temp", -40, 50)
    
    logger.info("Silver Chicago Taxi Enriched DQ tests Passed.")
except Exception as e:
    logger.error(f"DQ Failed: {str(e)}")
    dbutils.notebook.exit(str(e))

**Save Enriched And Preprocessed Chicago Taxi Data to Silver Layer**

In [0]:

try:
    logger.info(f"Writing df_silver_taxi_enriched to Silver Layer")
    df_silver_taxi_enriched.write\
        .format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true")\
        .partitionBy("year", "month")\
        .saveAsTable(silver_taxi_data_enriched)
    logger.info(f"Data written successfully to table `{silver_taxi_data_enriched}` Delta Table")
except Exception as e:
   logger.error(f"Error writing silver_taxi_data_enriched to Silver Layer: {e}")
   dbutils.notebook.exit(e)